# Part 7: Model Selection & Boosting

[← Back to Index](Index.ipynb)

**Quick Reference Guide for Hyperparameter Tuning and Boosting Algorithms**

---
## 7.1 Model Selection Techniques

**Purpose:** Find optimal hyperparameters to maximize model performance

### Grid Search

**What:** Exhaustive search over specified parameter values

**Pros:**
- Guarantees finding best combination (in grid)
- Simple and deterministic

**Cons:**
- Computationally expensive
- Exponential growth with parameters

**When to use:** Small parameter space, sufficient compute

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Load data
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define model
rf = RandomForestClassifier(random_state=42)

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
# Total combinations: 3 * 4 * 3 * 3 = 108

# Grid Search with CV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,                    # 5-fold cross-validation
    scoring='accuracy',      # can be 'f1', 'roc_auc', etc.
    n_jobs=-1,              # use all cores
    verbose=2,              # print progress
    return_train_score=True
)

# Fit (this will take time)
grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Test performance
test_score = grid_search.score(X_test, y_test)
print(f"Test Score: {test_score:.4f}")

# Access best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

In [ ]:
# Analyze results
results_df = pd.DataFrame(grid_search.cv_results_)

# View top 10 configurations
print("\nTop 10 Configurations:")
print(results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
      .sort_values('rank_test_score').head(10))

# Visualize parameter importance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

params = ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf']
for idx, param in enumerate(params):
    param_col = f'param_{param}'
    grouped = results_df.groupby(param_col)['mean_test_score'].mean().sort_index()
    
    axes[idx].plot(grouped.index.astype(str), grouped.values, 'o-')
    axes[idx].set_xlabel(param)
    axes[idx].set_ylabel('Mean CV Score')
    axes[idx].set_title(f'Effect of {param}')
    axes[idx].grid(True)

plt.tight_layout()
plt.show()

### Random Search

**What:** Random sampling from parameter distributions

**Pros:**
- Much faster than Grid Search
- Can specify budget (n_iter)
- Often finds good solutions quickly
- Better for continuous parameters

**Cons:**
- Not exhaustive
- May miss optimal combination

**When to use:** Large parameter space, limited compute

In [ ]:
from scipy.stats import randint, uniform

# Define parameter distributions
param_distributions = {
    'n_estimators': randint(50, 500),           # random integers
    'max_depth': [None] + list(range(5, 50)),   # mix of None and range
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': uniform(0.1, 0.9),         # continuous uniform
    'bootstrap': [True, False]
}

# Random Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=100,              # number of random samples
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    random_state=42,
    return_train_score=True
)

# Fit
random_search.fit(X_train, y_train)

# Results
print("Best Parameters:", random_search.best_params_)
print(f"Best CV Score: {random_search.best_score_:.4f}")
print(f"Test Score: {random_search.score(X_test, y_test):.4f}")

# Compare Grid vs Random
print(f"\nGrid Search: {grid_search.best_score_:.4f}")
print(f"Random Search: {random_search.best_score_:.4f}")

### Bayesian Optimization

**What:** Smart search using probabilistic model

**Pros:**
- Most efficient
- Learns from previous evaluations
- Great for expensive models

**Cons:**
- More complex
- Requires additional library

**When to use:** Expensive models (deep learning), limited budget

In [ ]:
# Install: pip install scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# Define search space
search_space = {
    'n_estimators': Integer(50, 500),
    'max_depth': Integer(5, 50),
    'min_samples_split': Integer(2, 20),
    'min_samples_leaf': Integer(1, 10),
    'max_features': Real(0.1, 1.0),
    'bootstrap': Categorical([True, False])
}

# Bayesian Optimization
bayes_search = BayesSearchCV(
    estimator=rf,
    search_spaces=search_space,
    n_iter=50,              # fewer iterations needed
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# Fit
bayes_search.fit(X_train, y_train)

# Results
print("Best Parameters:", bayes_search.best_params_)
print(f"Best CV Score: {bayes_search.best_score_:.4f}")
print(f"Test Score: {bayes_search.score(X_test, y_test):.4f}")

# Convergence plot
from skopt.plots import plot_convergence
plot_convergence(bayes_search.optimizer_results_[0])
plt.show()

**Comparison Table:**

| Method | Speed | Efficiency | Best Use |
|--------|-------|------------|----------|
| Grid Search | Slow | Low | Small parameter space |
| Random Search | Medium | Medium | Large parameter space |
| Bayesian | Fast | High | Expensive models, limited budget |

---
## 7.2 Cross Validation Methods

**Purpose:** Robust model evaluation and prevent overfitting

### K-Fold Cross Validation

**What:** Split data into K folds, train K times

**When to use:** Standard datasets, sufficient data

In [ ]:
from sklearn.model_selection import cross_val_score, cross_validate, KFold
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

# 1. Simple cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"CV Scores: {scores}")
print(f"Mean: {scores.mean():.4f} (+/- {scores.std():.4f})")

# 2. With custom KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
print(f"\nCustom KFold Mean: {scores.mean():.4f}")

# 3. Multiple metrics with cross_validate
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_results = cross_validate(model, X, y, cv=5, scoring=scoring, 
                           return_train_score=True)

# Display results
for metric in scoring:
    test_scores = cv_results[f'test_{metric}']
    print(f"{metric}: {test_scores.mean():.4f} (+/- {test_scores.std():.4f})")

In [ ]:
# Visualize CV folds
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=False)
fig, ax = plt.subplots(figsize=(12, 3))

for i, (train_idx, test_idx) in enumerate(kfold.split(X)):
    # Create color array
    colors = np.zeros(len(X))
    colors[train_idx] = 1
    colors[test_idx] = 2
    
    ax.scatter(range(len(X)), [i] * len(X), c=colors, cmap='RdYlGn', 
              marker='|', s=100, alpha=0.8)

ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Sample Index')
ax.set_title('K-Fold Cross Validation (Green=Train, Red=Test)')
plt.tight_layout()
plt.show()

### Stratified K-Fold

**What:** K-Fold with preserved class distribution

**When to use:** Imbalanced datasets, classification

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Stratified K-Fold (maintains class proportions)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skfold, scoring='accuracy')

print(f"Stratified K-Fold Mean: {scores.mean():.4f}")

# Compare class distributions
print("\nOriginal class distribution:", np.bincount(y) / len(y))

for i, (train_idx, test_idx) in enumerate(skfold.split(X, y)):
    train_dist = np.bincount(y[train_idx]) / len(train_idx)
    test_dist = np.bincount(y[test_idx]) / len(test_idx)
    print(f"Fold {i+1} - Train: {train_dist}, Test: {test_dist}")

### Time Series Cross Validation

**What:** Forward-chaining splits to prevent data leakage

**When to use:** Time series data, temporal dependencies

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Time Series Split
tscv = TimeSeriesSplit(n_splits=5)

# Visualize splits
fig, ax = plt.subplots(figsize=(12, 5))

for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    colors = np.zeros(len(X))
    colors[train_idx] = 1
    colors[test_idx] = 2
    
    ax.scatter(range(len(X)), [i] * len(X), c=colors, cmap='RdYlGn',
              marker='|', s=100, alpha=0.8)
    
    print(f"Fold {i+1}: Train size={len(train_idx)}, Test size={len(test_idx)}")

ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Time Index')
ax.set_title('Time Series Cross Validation (Training grows over time)')
plt.tight_layout()
plt.show()

# Use in cross_val_score
scores = cross_val_score(model, X, y, cv=tscv, scoring='accuracy')
print(f"\nTime Series CV Mean: {scores.mean():.4f}")

### Nested Cross Validation

**What:** Outer CV for model evaluation, inner CV for hyperparameter tuning

**When to use:** Unbiased performance estimate with hyperparameter tuning

In [ ]:
from sklearn.model_selection import cross_val_score

# Nested CV
# Inner loop: hyperparameter tuning (Grid Search)
# Outer loop: model evaluation

param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [10, 20, None]}
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Create GridSearchCV (inner loop)
clf = GridSearchCV(estimator=rf, param_grid=param_grid, cv=inner_cv)

# Outer CV evaluates the GridSearchCV
nested_scores = cross_val_score(clf, X, y, cv=outer_cv, scoring='accuracy')

print(f"Nested CV Scores: {nested_scores}")
print(f"Mean: {nested_scores.mean():.4f} (+/- {nested_scores.std():.4f})")

# Compare with non-nested
non_nested_scores = cross_val_score(rf, X, y, cv=outer_cv, scoring='accuracy')
print(f"\nNon-nested Mean: {non_nested_scores.mean():.4f}")
print(f"Nested Mean: {nested_scores.mean():.4f}")
print("\nNested CV gives unbiased estimate!")

**Other CV Methods:**

In [ ]:
from sklearn.model_selection import LeaveOneOut, LeavePOut, ShuffleSplit

# Leave-One-Out (LOO) - extreme but expensive
loo = LeaveOneOut()
scores = cross_val_score(model, X[:100], y[:100], cv=loo)  # use subset
print(f"LOO Mean: {scores.mean():.4f} (n_splits={loo.get_n_splits(X[:100])})")

# Leave-P-Out (LPO)
lpo = LeavePOut(p=2)
# Very expensive, use on small data only

# Shuffle Split (random train/test splits)
shuffle_split = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
scores = cross_val_score(model, X, y, cv=shuffle_split)
print(f"Shuffle Split Mean: {scores.mean():.4f}")

---
## 7.3 XGBoost

**What:** Extreme Gradient Boosting - optimized gradient boosting

**Key Features:**
- Regularization (L1/L2)
- Parallel processing
- Handles missing values
- Tree pruning
- Built-in CV

**When to use:** Tabular data, competitions, production

In [ ]:
# Install: pip install xgboost
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# Load data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. sklearn API (recommended for beginners)
xgb_clf = xgb.XGBClassifier(
    n_estimators=100,         # number of boosting rounds
    max_depth=6,              # tree depth
    learning_rate=0.1,        # eta, step size
    subsample=0.8,            # row sampling
    colsample_bytree=0.8,     # column sampling
    gamma=0,                  # min split loss
    reg_alpha=0,              # L1 regularization
    reg_lambda=1,             # L2 regularization
    random_state=42,
    eval_metric='logloss'     # evaluation metric
)

# Train
xgb_clf.fit(X_train, y_train)

# Predict
y_pred = xgb_clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# 2. Native API with early stopping
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'binary:logistic',  # binary classification
    'max_depth': 6,
    'learning_rate': 0.1,
    'eval_metric': 'logloss',
    'seed': 42
}

# Train with evaluation
evals = [(dtrain, 'train'), (dtest, 'test')]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    early_stopping_rounds=50,  # stop if no improvement
    verbose_eval=100           # print every 100 rounds
)

print(f"\nBest iteration: {model.best_iteration}")
print(f"Best score: {model.best_score:.4f}")

In [ ]:
# Feature importance
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
feature_names = data.feature_names

# Get importance
importance = xgb_clf.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

# Plot top 10
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'][:10], importance_df['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Built-in plot (using native API)
xgb.plot_importance(model, max_num_features=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation with XGBoost
cv_results = xgb.cv(
    params,
    dtrain,
    num_boost_round=1000,
    nfold=5,
    metrics='logloss',
    early_stopping_rounds=50,
    seed=42,
    verbose_eval=100
)

print(f"Best iteration: {len(cv_results)}")
print(f"Best score: {cv_results['test-logloss-mean'].min():.4f}")

# Plot learning curves
plt.figure(figsize=(10, 6))
plt.plot(cv_results.index, cv_results['train-logloss-mean'], label='Train')
plt.plot(cv_results.index, cv_results['test-logloss-mean'], label='Test')
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('XGBoost Learning Curves')
plt.legend()
plt.grid(True)
plt.show()

---
## 7.4 Advanced Boosting Methods

### LightGBM

**What:** Light Gradient Boosting Machine by Microsoft

**Advantages:**
- Faster training
- Lower memory usage
- Better accuracy
- Handles large datasets
- Leaf-wise growth (vs level-wise in XGBoost)

**When to use:** Large datasets, need speed

In [ ]:
# Install: pip install lightgbm
import lightgbm as lgb

# 1. sklearn API
lgb_clf = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,             # no limit (leaf-wise)
    num_leaves=31,            # max leaves per tree
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,              # L1
    reg_lambda=0,             # L2
    random_state=42
)

lgb_clf.fit(X_train, y_train)
y_pred = lgb_clf.predict(X_test)
print(f"LightGBM Accuracy: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# 2. Native API
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.1,
    'feature_fraction': 0.8,
    'verbose': -1
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# Feature importance
lgb.plot_importance(model, max_num_features=10)
plt.tight_layout()
plt.show()

### CatBoost

**What:** Categorical Boosting by Yandex

**Advantages:**
- Native categorical feature handling
- Robust to overfitting
- No need for extensive tuning
- GPU support

**When to use:** Many categorical features

In [ ]:
# Install: pip install catboost
from catboost import CatBoostClassifier, Pool

# CatBoost
cat_clf = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=3,           # L2 regularization
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

# Train with early stopping
cat_clf.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=100
)

y_pred = cat_clf.predict(X_test)
print(f"\nCatBoost Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Feature importance
feature_importance = cat_clf.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'][:10], importance_df['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Features (CatBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Handling categorical features (CatBoost's strength)
# Example with categorical data
df = pd.DataFrame({
    'num1': np.random.randn(1000),
    'num2': np.random.randn(1000),
    'cat1': np.random.choice(['A', 'B', 'C'], 1000),
    'cat2': np.random.choice(['X', 'Y', 'Z'], 1000),
    'target': np.random.randint(0, 2, 1000)
})

X = df.drop('target', axis=1)
y = df['target']

# Specify categorical features (no need to encode!)
cat_features = ['cat1', 'cat2']

cat_clf = CatBoostClassifier(
    iterations=100,
    cat_features=cat_features,  # specify categorical columns
    verbose=0
)

cat_clf.fit(X, y)
print("CatBoost handles categorical features automatically!")

### AdaBoost

**What:** Adaptive Boosting (classic algorithm)

**When to use:** Baseline boosting, educational purposes

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# AdaBoost with Decision Trees
ada_clf = AdaBoostClassifier(
    base_estimator=DecisionTreeClassifier(max_depth=1),  # weak learners
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

ada_clf.fit(X_train, y_train)
y_pred = ada_clf.predict(X_test)
print(f"AdaBoost Accuracy: {accuracy_score(y_test, y_pred):.4f}")

### Gradient Boosting (sklearn)

**What:** Original gradient boosting implementation

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gb_clf.fit(X_train, y_train)
y_pred = gb_clf.predict(X_test)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_test, y_pred):.4f}")

**Boosting Comparison:**

| Algorithm | Speed | Memory | Categorical | Best For |
|-----------|-------|--------|-------------|----------|
| XGBoost | Fast | Medium | Manual encoding | General purpose, competitions |
| LightGBM | Fastest | Low | Manual encoding | Large datasets |
| CatBoost | Medium | High | Native support | Many categorical features |
| AdaBoost | Slow | Low | Manual encoding | Simple problems |
| GradientBoosting | Slow | Medium | Manual encoding | Small datasets |

---
## 7.5 Ensemble Methods

**Purpose:** Combine multiple models for better performance

### Voting Classifier

**What:** Combine predictions from multiple models

**Types:**
- Hard Voting: Majority vote
- Soft Voting: Average probabilities

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Create base models
log_clf = LogisticRegression(max_iter=1000, random_state=42)
svm_clf = SVC(probability=True, random_state=42)  # probability=True for soft voting
dt_clf = DecisionTreeClassifier(random_state=42)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Hard Voting
voting_hard = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('svm', svm_clf),
        ('dt', dt_clf),
        ('rf', rf_clf)
    ],
    voting='hard'
)

voting_hard.fit(X_train, y_train)
print(f"Hard Voting Accuracy: {voting_hard.score(X_test, y_test):.4f}")

# Soft Voting (usually better)
voting_soft = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('svm', svm_clf),
        ('dt', dt_clf),
        ('rf', rf_clf)
    ],
    voting='soft'
)

voting_soft.fit(X_train, y_train)
print(f"Soft Voting Accuracy: {voting_soft.score(X_test, y_test):.4f}")

# Compare individual models
print("\nIndividual Model Scores:")
for name, clf in voting_hard.named_estimators_.items():
    print(f"{name}: {clf.score(X_test, y_test):.4f}")

### Bagging vs Boosting

**Bagging (Bootstrap Aggregating):**
- Parallel training
- Reduces variance
- Example: Random Forest

**Boosting:**
- Sequential training
- Reduces bias
- Example: XGBoost, AdaBoost

In [ ]:
from sklearn.ensemble import BaggingClassifier

# Bagging with Decision Trees
bagging_clf = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,         # sample 80% of data
    max_features=0.8,        # sample 80% of features
    bootstrap=True,          # sample with replacement
    n_jobs=-1,
    random_state=42
)

bagging_clf.fit(X_train, y_train)
print(f"Bagging Accuracy: {bagging_clf.score(X_test, y_test):.4f}")

# Compare with single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print(f"Single Tree Accuracy: {dt.score(X_test, y_test):.4f}")
print("\nBagging reduces variance and improves stability!")

### Stacking

**What:** Use meta-model to combine base model predictions

**How:**
1. Train base models on data
2. Use base predictions as features
3. Train meta-model on these features

In [ ]:
from sklearn.ensemble import StackingClassifier

# Base models (level 0)
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(probability=True, random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

# Meta model (level 1)
meta_model = LogisticRegression()

# Stacking
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,  # cross-validation to generate meta-features
    n_jobs=-1
)

stacking_clf.fit(X_train, y_train)
print(f"Stacking Accuracy: {stacking_clf.score(X_test, y_test):.4f}")

# Compare with base models
print("\nBase Model Scores:")
for name, model in base_models:
    model.fit(X_train, y_train)
    print(f"{name}: {model.score(X_test, y_test):.4f}")

### Blending

**What:** Similar to stacking but uses hold-out set

**Difference from Stacking:**
- Stacking: Uses CV to generate meta-features
- Blending: Uses hold-out validation set

In [ ]:
# Manual Blending
# Split into train, validation, test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42)

# Train base models on train set
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
svm_clf = SVC(probability=True, random_state=42)
lr_clf = LogisticRegression(max_iter=1000, random_state=42)

rf_clf.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)
lr_clf.fit(X_train, y_train)

# Get predictions on validation set (meta-features)
val_pred_rf = rf_clf.predict_proba(X_val)[:, 1]
val_pred_svm = svm_clf.predict_proba(X_val)[:, 1]
val_pred_lr = lr_clf.predict_proba(X_val)[:, 1]

# Stack predictions
X_val_meta = np.column_stack([val_pred_rf, val_pred_svm, val_pred_lr])

# Train meta-model on validation predictions
meta_model = LogisticRegression()
meta_model.fit(X_val_meta, y_val)

# Test: get base predictions on test set
test_pred_rf = rf_clf.predict_proba(X_test)[:, 1]
test_pred_svm = svm_clf.predict_proba(X_test)[:, 1]
test_pred_lr = lr_clf.predict_proba(X_test)[:, 1]
X_test_meta = np.column_stack([test_pred_rf, test_pred_svm, test_pred_lr])

# Final prediction
y_pred = meta_model.predict(X_test_meta)
print(f"Blending Accuracy: {accuracy_score(y_test, y_pred):.4f}")

---
### Quick Reference Guide

**Hyperparameter Tuning:**
- Small search space: Grid Search
- Large search space: Random Search
- Expensive models: Bayesian Optimization

**Cross Validation:**
- Standard: K-Fold
- Imbalanced: Stratified K-Fold
- Time series: TimeSeriesSplit
- Unbiased estimate: Nested CV

**Boosting:**
- General purpose: XGBoost
- Large data: LightGBM
- Many categorical: CatBoost

**Ensemble:**
- Quick ensemble: Voting
- Reduce variance: Bagging
- Reduce bias: Boosting
- Best performance: Stacking

**Best Practices:**
- Always use CV for hyperparameter tuning
- Start with Random Search, then Grid Search around best
- Use early stopping with boosting
- Monitor train/validation curves for overfitting
- Ensemble diverse models for best results

---
[← Back to Index](Index.ipynb)